# Tools and Agents

In [19]:
import os
import json

from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_DEFAULT_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "ibm-granite/granite-4.1-8b"

from langchain_openrouter import ChatOpenRouter 

from langchain.tools import tool
from langchain_classic.agents import create_react_agent, AgentExecutor

from langchain_core.tools import Tool
from langchain_core.prompts import PromptTemplate

from langchain_experimental.utilities import PythonREPL


def llm_model(params=None):

    # 1. Define sensible defaults
    config = {
        "model": OPENROUTER_MODEL,
        "api_key": OPENROUTER_API_KEY,
        "base_url": OPENROUTER_DEFAULT_BASE_URL,
        "temperature": 0.5,
        "max_tokens": 256,
        "max_completion_tokens": 128
    }
    
    if params:
        config.update(params)
        
    # 3. Initialize the model
    model = ChatOpenRouter(
        model=config["model"],
        api_key=config["api_key"],
        base_url=config["base_url"],
        temperature=config["temperature"],
        max_tokens=config["max_tokens"],
        max_completion_tokens=config["max_completion_tokens"]
    )

    return model

def llm_model_response(prompt_text, params=None):
            
    # 3. Initialize the model
    model = llm_model(params)

    response = model.invoke(prompt_text)

    return response

## Tools

In [6]:
# Create a PythonREPL instance
# This provides an environment where Python code can be executed as strings
python_repl = PythonREPL()

# Create a Tool using the Tool class
# This wraps the Python REPL functionality as a tool that can be used by agents
python_calculator = Tool(
    # The name of the tool - this helps agents identify when to use this tool
    name="Python Calculator",
    
    # The function that will be called when the tool is used
    # python_repl.run takes a string of Python code and executes it
    func=python_repl.run,
    
    # A description of what the tool does and how to use it
    # This helps the agent understand when and how to use this tool
    description="Useful for when you need to perform calculations or execute Python code. Input should be valid Python code."
)

python_calculator.invoke("a = 3; b = 1; print(a+b)")

@tool
def search_weather(location: str):
    """Search for the current weather in the specified location."""
    # In a real application, this would call a weather API
    return f"The weather in {location} is currently sunny and 72°F."

## Toolkit

In [7]:
# Create a toolkit (collection of tools)
tools = [python_calculator, search_weather]

## Agents

In [24]:
# Create the ReAct agent prompt template
# The ReAct prompt needs to instruct the model to follow the thought-action-observation pattern
prompt_template = """You are an agent who has access to the following tools:
{tools}
The available tools are: {tool_names}
To use a tool, please use the following format:
```
Thought: I need to figure out what to do
Action: tool_name
Action Input: the input to the tool
```
After you use a tool, the observation will be provided to you:
```
Observation: result of the tool
```
Then you should continue with the thought-action-observation cycle until you have enough information to respond to the user's request directly.
When you have the final answer, respond in this format:
```
Thought: I know the answer
Final Answer: the final answer to the original query
```
Remember, when using the Python Calculator tool, the input must be valid Python code.
Begin!

Question: {input}
{agent_scratchpad}
"""

prompt = PromptTemplate.from_template(prompt_template)


OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

agent = create_react_agent(
    llm=model,
    tools=tools,
    prompt=prompt
)

# Create the agent executor
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,
    handle_parsing_errors=True
)


### Test the agent with a simple problem

In [ ]:
# Ask the agent a question that requires only calculation
result = agent_executor.invoke({"input": "What is the square root of 256?"})
print(result["output"])

### Test the agent with different types of queries

In [ ]:
# Examples of different types of queries to test the agent
queries = [
    "What's 345 * 789?",
    "Calculate the square root of 144",
    "What's the weather in Miami?",
    "If it's sunny in Chicago, what would be a good outdoor activity?",
    "Generate a list of prime numbers below 50 and calculate their sum"
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print(f"{'='*60}")
    
    result = agent_executor.invoke({"input": query})
    
    print(f"\nFINAL ANSWER: {result['output']}")

### Creating Your First LangChain Agent with Basic Tools

In [29]:
# Create a simple calculator tool
def calculator(expression: str) -> str:
    """A simple calculator that can add, subtract, multiply, or divide two numbers.
    Input should be a mathematical expression like '2 + 2' or '15 / 3'."""
    try:
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"

# Create a text formatting tool
def format_text(text: str) -> str:
    """Format text to uppercase, lowercase, or title case.
    Input should be in format: [format_type]: [text]
    where format_type is uppercase, lowercase, or titlecase.
    
    Examples:
    - uppercase: hello world -> HELLO WORLD
    - lowercase: HELLO WORLD -> hello world 
    - titlecase: hello world -> Hello World
    """
    try:
        # Handle the case where the entire string is passed
        if ":" in text:
            format_type, content = text.split(":", 1)
            format_type = format_type.strip().lower()
            content = content.strip()
        else:
            # If no colon, assume they want titlecase
            return f"Missing format. Example: titlecase: {text} -> {text.title()}"
            
        if format_type == "uppercase":
            return content.upper()
        elif format_type == "lowercase":
            return content.lower()
        elif format_type == "titlecase":
            return content.title()
        else:
            return f"Unknown format {format_type}. Use: uppercase, lowercase, or titlecase"
            
    except Exception as e:
        return f"Error formatting text: {str(e)}"

OPENROUTER_MODEL = "openai/gpt-4o-mini"

params = {
    "model": OPENROUTER_MODEL,
    "api_key": OPENROUTER_API_KEY,
    "base_url": OPENROUTER_DEFAULT_BASE_URL,
    "temperature": 0.8,
    "max_tokens": 1024,
    "max_completion_tokens": 512
}

model = llm_model(params=params)

# Create Tool objects for our functions
tools = [
    Tool(
        name="calculator",
        func=calculator,
        description="Useful for performing simple math calculations"
    ),
    Tool(
        name="format_text",
        func=format_text,
        description="Useful for formatting text to uppercase, lowercase, or titlecase"
    )
]

# Create a simple prompt template
# Note the added {tool_names} variable which was missing before
prompt_template = """You are a helpful assistant who can use tools to help with simple tasks.
You have access to these tools:
{tools}
The available tools are: {tool_names}
Follow this format:
Question: the user's question
Thought: think about what to do
Action: the tool to use, should be one of [{tool_names}]
Action Input: the input to the tool
Observation: the result from the tool
Thought: I now know the final answer
Final Answer: your final answer to the user's question
Question: {input}
{agent_scratchpad}
"""

# Create the agent and executor
prompt = PromptTemplate.from_template(prompt_template)
agent = create_react_agent(
    llm=model,
    tools=tools,
    prompt=prompt
)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# Test with simple questions
test_questions = [
    "What is 25 + 63?", # The agent will be able to answer this question
    "Can you convert 'hello world' to uppercase?" # The agent might be able to answer this question
                                                    # However, it is not guaranteed due to incorrect input format
    "Calculate 15 * 7", # The agent will be able to answer this question
    "titlecase: langchain is awesome", # The agent will be able to answer this question
]

# Run the tests
for question in test_questions:
    print(f"\n===== Testing: {question} =====")
    result = agent_executor.invoke({"input": question})
    print(f"Final Answer: {result['output']}")





===== Testing: Can you convert 'hello world' to uppercase? =====


> Entering new AgentExecutor chain...
Thought: I need to convert the text 'hello world' to uppercase.  
Action: format_text  
Action Input: 'hello world'  Missing format. Example: titlecase: 'hello world' -> 'Hello World'I need to specify that I want to convert the text to uppercase.  
Action: format_text  
Action Input: 'hello world'.upper()  

KeyboardInterrupt: 